In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns


## Setup

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Load paths from .env file
dataset_dir = os.getenv("DATASET_DIR")
datasplits_dir = os.getenv("DATASPLITS_DIR")

# Patgs to the data splits files
train_file = os.path.join(datasplits_dir, "img_train.tsv")
val_file = os.path.join(datasplits_dir, "img_val.tsv")
test_file = os.path.join(datasplits_dir, "img_test.tsv")

# Load the data splits into pandas DataFrames
train_df = pd.read_csv(train_file, sep="\t")
val_df = pd.read_csv(val_file, sep="\t")
test_df = pd.read_csv(test_file, sep="\t")

In [ ]:
train_df.shape

In [ ]:
train_df.head(1)

In [ ]:
val_df.shape

In [ ]:
val_df.head(1)

In [ ]:
test_df.shape

In [ ]:
test_df.head(1)

## Hyperparameters

In [ ]:
BATCH_SIZE = 32

## Label Encoding

In [ ]:
label_encoders = {}
for col in ["image_info", "image_human"]:
	le = LabelEncoder()
	all_values = pd.concat([train_df[col], val_df[col], test_df[col]])
	le.fit(all_values)
	train_df[col] = le.transform(train_df[col])
	val_df[col] = le.transform(val_df[col])
	test_df[col] = le.transform(test_df[col])
	label_encoders[col] = le


## Data Loading

In [ ]:
IMAGE_SIZE = (224, 224)

def load_image(path, label1, label2):
	img = tf.io.read_file(path)
	img = tf.image.decode_jpeg(img, channels=3)
	img = tf.image.resize(img, IMAGE_SIZE)
	img = preprocess_input(img)
	return img, {"info": label1, "human": label2}

def df_to_dataset(df, shuffle=True):
	full_paths = df["image_path"].apply(lambda p: os.path.normpath(os.path.join(dataset_dir, p))).values
	labels1 = df["image_info"].values.astype("float32")  # for sigmoid binary output
	labels2 = df["image_human"].values
	ds = tf.data.Dataset.from_tensor_slices((full_paths, labels1, labels2))
	ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
	if shuffle:
		ds = ds.shuffle(700)
	return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = df_to_dataset(train_df)
val_ds = df_to_dataset(val_df, shuffle=False)
test_ds = df_to_dataset(test_df, shuffle=False)

## Model definition

In [ ]:
base_model = tf.keras.applications.ResNet50(
	input_shape=(*IMAGE_SIZE, 3),
	include_top=False,
	weights="imagenet"
)

In [ ]:
base_model.summary()

In [ ]:
base_model.trainable = False  # freeze
print("This is the number of trainable weights after freezing the conv base:", len(base_model.trainable_weights))

In [ ]:
data_augmentation = tf.keras.Sequential([
	tf.keras.layers.RandomFlip("horizontal"),
	tf.keras.layers.RandomRotation(0.1),
	tf.keras.layers.RandomZoom(0.1),
])

inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3))
x = data_augmentation(inputs)
x = base_model(x, training=True)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)

# Output heads
info_out = tf.keras.layers.Dense(1, activation="sigmoid", name="info")(x)
human_out = tf.keras.layers.Dense(train_df["image_human"].nunique(), activation="softmax", name="human")(x)

model = tf.keras.Model(inputs, outputs=[info_out, human_out])

## Model compile

In [ ]:
model.compile(
	optimizer="adam",
	loss={
		"info": "binary_crossentropy",
		"human": "sparse_categorical_crossentropy"
	},
	metrics={
		"info": "accuracy",
		"human": "accuracy"
	}
)

## Training

In [ ]:
callbacks = [
	ModelCheckpoint("models/best_resnet_model.keras", monitor="val_loss", save_best_only=True, verbose=1),
	EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)
]

history = model.fit(
	train_ds,
	epochs=20,
	validation_data=val_ds,
	callbacks=callbacks)


In [ ]:
def plot_training_history(history):
	for key in history.history:
		if not key.startswith("val_"):
			plt.figure()
			plt.plot(history.history[key], label="train")
			plt.plot(history.history[f"val_{key}"], label="val")
			plt.title(key)
			plt.xlabel("Epoch")
			plt.ylabel("Value")
			plt.legend()
			plt.grid(True)
			plt.show()

plot_training_history(history)


## Fine-tuning

In [ ]:
train_ds = df_to_dataset(train_df)
val_ds = df_to_dataset(val_df, shuffle=False)

In [ ]:
# Unfreeze last block (conv5_block)
for layer in base_model.layers:
    if layer.name.startswith("conv5_block"):
        layer.trainable = True

print("Trainable layers after partial unfreezing:", sum([layer.trainable for layer in base_model.layers]))

In [ ]:
model.load_weights("./models/best_resnet_model.keras")

In [ ]:
callbacks = [
	ModelCheckpoint("./models/fine_tuned_resnet_model.keras", monitor="val_loss", save_best_only=True, verbose=1)
]

history = model.fit(
	train_ds,
	epochs=5,
	validation_data=val_ds,
	callbacks=callbacks)

In [ ]:
plot_training_history(history)

## Evaluation

In [ ]:
model.load_weights("models/fine_tuned_resnet_model.keras")

In [ ]:
# Initialize lists to collect ground-truth and predictions
y_true_info, y_true_human = [], []
y_pred_info, y_pred_human = [], []

# Add progress bar to prediction loop
for x_batch, y_batch in tqdm(test_ds, desc="Running inference on test set"):
	# Predict outputs for each task
	preds = model.predict(x_batch, verbose=0)
	
	# Task 1: informativeness (binary - sigmoid)
	y_true_info.extend(y_batch["info"].numpy())
	y_pred_info.extend((preds[0] > 0.5).astype("int32").flatten())

	# Task 2: humanitarian category (multiclass - softmax)
	y_true_human.extend(y_batch["human"].numpy())
	y_pred_human.extend(np.argmax(preds[1], axis=1))


In [ ]:
def evaluate_task(y_true, y_pred, task_name, class_names=None):
    print(f"\n📊 {task_name.upper()} classification report:")

    print(classification_report(y_true, y_pred, target_names=class_names))

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

    print(f"✅ Accuracy:  {acc:.4f}")
    print(f"✅ Precision: {prec:.4f}")
    print(f"✅ Recall:    {rec:.4f}")
    print(f"✅ F1-score:  {f1:.4f}")

    # Shorten class names only for "image_human" task
    if task_name == "image_human":
        display_names = [c[0].upper() for c in class_names]
    else:
        display_names = class_names

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap="Blues",
                xticklabels=display_names, yticklabels=display_names)
    plt.title(f"{task_name.upper()} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()


In [ ]:
# TASK 1: Image Information Classification Evaluation
evaluate_task(
	y_true_info,
	y_pred_info,
	"image_info",
	class_names=["not informative", "informative"]
)

In [ ]:
# TASK 2: Humanitarian Classification Evaluation
evaluate_task(
	y_true_human,
	y_pred_human,
	"image_human",
	class_names=label_encoders["image_human"].classes_
)